# Week 3: Digital Modulation I - From Bits to Constellation Points

ECE 5377/7377 Embedded Wireless Design Laboratory

This Python notebook replaces the Week 3 MATLAB demonstration. It follows the slide, run-of-show, and whiteboard sequence: bits per symbol, 4-PAM mapping, I/Q constellations, mean, symbol energy, normalization, and noisy constellation clouds.

---
## Instructor companion: reading this notebook at the podium

The original section sequence, executable statements, numeric settings, plotting calls, function docstrings, and existing Markdown remain intact. Added material is marked **Instructor guide** or appears as Python `#` comments. It explains what the supplied demonstration actually computes, including its assumptions and limitations. It is not an additional lab assignment.

**Prepare:** run the notebook from the top in a fresh kernel, then save it with the outputs you intend to show. Run earlier setup cells before later demonstrations. A function definition installs the helper; its body executes only when called. The bracketed Jupyter execution number is not a section number. Random-number generators retain state, so rerunning only a noise cell gives a new realization.

**During class:** make the prediction, run the cell, interpret the result. It is not necessary to teach every plotting command. The purpose, expected result, and variable table before each block are there for quick reference. Names are case-sensitive: `x` and `X` are different variables; the same short name can be reused for a different purpose in another section or helper.

### C/C++ translation reminders

| Python / NumPy form | Read it this way |
|---|---|
| `1j`, `.real`, `.imag` | Complex unit; real and imaginary coordinates. Similar to `std::complex<double>` operations. |
| `np.abs(z)` / `np.abs(z)**2` | Complex magnitude / squared magnitude, not just the real component. |
| `a * b` versus `A @ B` | Elementwise multiplication (including scalar broadcasting) versus matrix multiplication. |
| `**` | Exponentiation. Python `^` would be bitwise XOR, not a power. |
| `a[start:stop:step]` | Select a slice; `stop` is excluded, and indices are zero-based. |
| `a[::-1]`, `a[-L:]` | Reverse order; select the last `L` entries. A negative index counts from the end. |
| `A[row, col]`, `A[:, j]` | Two-dimensional indexing; all rows of column `j`. |
| `reshape(-1, 1)`, `reshape(1, -1)` | Make an explicit column / row; infer the `-1` dimension. |
| `x[:, None]` | Add a length-one dimension so broadcasting can compare many samples with many candidates. |
| `.T`, `.conj().T` | Ordinary transpose; conjugate (Hermitian) transpose. They differ for complex data. |
| `for i, value in enumerate(a)` | Iterate over both index and value. `zip(a,b)` visits corresponding pairs. |
| `u, v = helper(...)` | Unpack two returned results. `_` is an ordinary name often used for a result intentionally ignored. |
| `np.mean(...)`, `np.sum(...)` | Average or sum the indicated values. An `axis` chooses which array dimension is reduced. |
| `argmin`, `argmax` | Return the index of the winning value, not the value itself. |
| `f"{value:.3f}"` | Format text with three decimal places. It changes display, not the stored value. |

Useful inspection commands, typed in a separate temporary code cell:

```python
%whos                 # list variables in this notebook's kernel
```

For an existing array, inspect its `.shape`, `.dtype`, or a short slice such as `x[:5]`. Local variables inside a function are not available in the notebook's global workspace after the function returns. These inspection examples are explanatory Markdown, not new executable cells.


## Setup


---
### Instructor guide: Setup

**Purpose:** Import the numerical and plotting libraries and reset the pseudorandom generator to the original fixed seed.

**Ask before running:** This cell prepares the notebook. Function definitions do not run their numerical bodies until a later cell calls them.

**What to expect:** A printed heading confirms execution. Run this cell first; later cells reuse its imports, functions, and generator.

**Teaching notes / model boundaries:** Use symbol modulation for bits-to-constellation mapping. No RF carrier or upconversion is simulated in this notebook. Here each complex number is one baseband symbol, not a sampled RF waveform.

#### Variable reference for this block

| Variable | Meaning / units / role in this block |
|---|---|
| `np` | Alias for the NumPy module, not an array. |
| `plt` | Alias for matplotlib.pyplot, the plotting interface. |
| `rng` | Stateful pseudorandom-number generator. Rerunning a later noise cell advances its state; restarting and running top-to-bottom reproduces the original sequence. |

**Live-demo cue:** Explain the prediction and result first. Use the detailed comments below only when a code operation needs clarification.


In [ ]:
# ========================================================================================
# INSTRUCTOR GUIDE | Setup
# Purpose: Import the numerical and plotting libraries and reset the pseudorandom generator to
# the original fixed seed.
# Original executable statements below are unchanged. Expanded variable explanations appear in
# the Markdown immediately above this cell.
# Read the calculation comments as preparation; the live demo can focus on prediction and the
# resulting plot.
# ========================================================================================

# Alias for the NumPy module, not an array.
import numpy as np
# Alias for matplotlib.pyplot, the plotting interface.
import matplotlib.pyplot as plt

# Use a fixed random-number generator so the demonstration is repeatable.
# rng: Stateful pseudorandom-number generator. Rerunning a later noise cell advances its
# state; restarting and running top-to-bottom reproduces the original sequence.
rng = np.random.default_rng(7)

# Assign only the indexed entries; the remaining array values are unchanged.
plt.rcParams["figure.figsize"] = (8, 4.8)
# Assign only the indexed entries; the remaining array values are unchanged.
plt.rcParams["axes.grid"] = True

# Print a teaching label or takeaway; no signal-processing state changes here.
print("WEEK 3: DIGITAL MODULATION I")

## 1. Modulation order and bits per symbol


---
### Instructor guide: 1. Modulation order and bits per symbol

**Purpose:** Translate the number of available symbol choices into the number of bits each choice can encode.

**Ask before running:** How many bits are needed to label 16 choices? How about 256?

**What to expect:** The printed pairs are 2 -> 1, 4 -> 2, 8 -> 3, 16 -> 4, 64 -> 6, and 256 -> 8.

**Teaching notes / model boundaries:** These M values are powers of two, so log2(M) is an integer before the type conversion. This is bits per symbol, not bits per second; a symbol rate would also be needed for a bit rate.

#### Variable reference for this block

| Variable | Meaning / units / role in this block |
|---|---|
| `M_values` | Array of constellation sizes: [2, 4, 8, 16, 64, 256]. |
| `bits_per_symbol` | Elementwise log2(M_values), converted to integers. |
| `M` | Current constellation size while printing the table. |
| `b` | Current bits-per-symbol value paired with M. |

**Live-demo cue:** Explain the prediction and result first. Use the detailed comments below only when a code operation needs clarification.


In [ ]:
# ========================================================================================
# INSTRUCTOR GUIDE | 1. Modulation order and bits per symbol
# Purpose: Translate the number of available symbol choices into the number of bits each
# choice can encode.
# Original executable statements below are unchanged. Expanded variable explanations appear in
# the Markdown immediately above this cell.
# Read the calculation comments as preparation; the live demo can focus on prediction and the
# resulting plot.
# ========================================================================================

# 1. Modulation order and bits per symbol

# M_values: Array of constellation sizes: [2, 4, 8, 16, 64, 256].
M_values = np.array([2, 4, 8, 16, 64, 256])
# np.log2 operates on the entire array. astype(int) converts its results to integers; it does
# not change the modulation order.
# C++ mental model: bits_per_symbol[i] = static_cast<int>(log2(M_values[i])).
bits_per_symbol = np.log2(M_values).astype(int)

# Print a teaching label or takeaway; no signal-processing state changes here.
print("MODULATION ORDER AND BITS PER SYMBOL")
# zip pairs corresponding entries of two arrays. Each iteration unpacks one pair into M and b.
# The f-string below formats M in a 3-character integer field; it does not change M.
for M, b in zip(M_values, bits_per_symbol):
    # Report calculated values. f-string precision/sign specifiers affect text only, not the
    # stored numbers.
    print(f"M = {M:3d} -> b = log2(M) = {b:d} bits/symbol")

## 2. Map a bit sequence into 4-PAM symbols


---
### Instructor guide: 2. Map bit pairs into 4-PAM

**Purpose:** Group eight bits into four pairs and use the explicit course mapping to select amplitudes.

**Ask before running:** For 00, 11, 10, 01, predict the four amplitude levels before running.

**What to expect:** The sequence is [-3, +1, +3, -1]. The horizontal axis is symbol index, not time in seconds.

**Teaching notes / model boundaries:** This is symbol mapping only. The stem plot is not yet a pulse-shaped continuous waveform. The adjacent amplitude labels differ by one bit, as specified by the supplied Gray mapping.

#### Variable reference for this block

| Variable | Meaning / units / role in this block |
|---|---|
| `mapping_4pam` | Python dictionary mapping a two-bit tuple to one amplitude level. |
| `bits` | Eight input bits: [0, 0, 1, 1, 1, 0, 0, 1]. |
| `bit_pairs` | Shape (4, 2): one two-bit group per row. |
| `pam_symbols` | Four real-valued 4-PAM symbols: [-3, 1, 3, -1]. |
| `pair` | Temporary row of two bits in the list comprehension. |
| `b` | Temporary bit used only while constructing printed text. |
| `s` | Temporary symbol used only while constructing printed text. |

**Live-demo cue:** Explain the prediction and result first. Use the detailed comments below only when a code operation needs clarification.


In [ ]:
# ========================================================================================
# INSTRUCTOR GUIDE | 2. Map bit pairs into 4-PAM
# Purpose: Group eight bits into four pairs and use the explicit course mapping to select
# amplitudes.
# Original executable statements below are unchanged. Expanded variable explanations appear in
# the Markdown immediately above this cell.
# Read the calculation comments as preparation; the live demo can focus on prediction and the
# resulting plot.
# ========================================================================================

# 2. Map a bit sequence into 4-PAM symbols
# Course mapping:
#   00 -> -3
#   01 -> -1
#   11 -> +1
#   10 -> +3

# A dictionary is a lookup table, similar to a C++ map. Tuple keys such as (0, 0) are
# immutable, so they can identify entries.
mapping_4pam = {
    (0, 0): -3,
    (0, 1): -1,
    (1, 1): +1,
    (1, 0): +3,
}

# bits: Eight input bits: [0, 0, 1, 1, 1, 0, 0, 1].
bits = np.array([0, 0,  1, 1,  1, 0,  0, 1])
# reshape(-1, 2) makes two columns and infers the number of rows: eight bits become four
# pairs.
# It reorganizes the same ordered values; it does not generate new bits.
bit_pairs = bits.reshape(-1, 2)
# This list comprehension replaces a loop: visit each row, convert it to a tuple, look up its
# amplitude, and collect the results.
# C++ mental model: for each pair, push_back(mapping[pair]); np.array converts that list to a
# numerical array.
pam_symbols = np.array([mapping_4pam[tuple(pair)] for pair in bit_pairs])

# Print a teaching label or takeaway; no signal-processing state changes here.
print("4-PAM MAPPING EXAMPLE")
# Build a human-readable report. join combines strings; it does not combine signal samples.
print("Input bits:     ", "".join(str(b) for b in bits))
# Build a human-readable report. join combines strings; it does not combine signal samples.
print("Grouped bits:   ", "   ".join("".join(str(b) for b in pair) for pair in bit_pairs))
# Build a human-readable report. join combines strings; it does not combine signal samples.
print("Mapped symbols: ", " ".join(f"{s:+.0f}" for s in pam_symbols))

# Create the figure/panels used for display; figsize specifies physical width and height in
# inches.
plt.figure()
# Draw individual discrete values with stems; the data are not assumed continuous between
# sample positions.
plt.stem(np.arange(len(pam_symbols)), pam_symbols)
plt.axhline(0)
plt.xlabel("Symbol index n")
plt.ylabel("4-PAM amplitude")
plt.title("4-PAM Symbols Produced from 00 11 10 01")
# Display the completed figure in Jupyter; this does not compute another transform or change
# the data.
plt.show()

## 3. Define and plot common constellations


---
### Instructor guide: 3. Define common constellations

**Purpose:** Show the complete sets of allowed BPSK, QPSK, and 16-QAM symbols in the I/Q plane.

**Ask before running:** How many allowed points appear for each modulation, and which have nonzero Q?

**What to expect:** BPSK has 2 points on the I axis; QPSK has 4 square-corner points; 16-QAM has a 4-by-4 grid.

**Teaching notes / model boundaries:** The constellation is the alphabet of possible symbols, not a finite random transmission. Its ordering matters for indexing but does not change the set of plotted points. These are raw, not yet equal-energy, constellations.

#### Variable reference for this block

| Variable | Meaning / units / role in this block |
|---|---|
| `C_bpsk` | Two allowed BPSK symbols; complex dtype permits an I/Q representation with Q = 0. |
| `C_qpsk_raw` | Four unnormalized QPSK points, with I and Q each equal to +/-1. |
| `levels` | Allowed raw amplitudes on either 16-QAM axis: [-3, -1, 1, 3]. |
| `C_qam16_raw` | Sixteen complex points from every combination of the I and Q levels. |
| `i` | Temporary in-phase coordinate in the constellation comprehension. |
| `q` | Temporary quadrature coordinate in the constellation comprehension. |
| `fig` | Figure containing the three panels; plotting object only. |
| `ax` | Array of three Axes objects, indexed 0, 1, 2. |
| `a` | Current Axes object in the formatting loop; not a symbol amplitude. |

**Live-demo cue:** Explain the prediction and result first. Use the detailed comments below only when a code operation needs clarification.


In [ ]:
# ========================================================================================
# INSTRUCTOR GUIDE | 3. Define common constellations
# Purpose: Show the complete sets of allowed BPSK, QPSK, and 16-QAM symbols in the I/Q plane.
# Original executable statements below are unchanged. Expanded variable explanations appear in
# the Markdown immediately above this cell.
# Read the calculation comments as preparation; the live demo can focus on prediction and the
# resulting plot.
# ========================================================================================

# 3. Define common constellations

# BPSK
# C_bpsk: Two allowed BPSK symbols; complex dtype permits an I/Q representation with Q = 0.
C_bpsk = np.array([-1, +1], dtype=complex)

# Unnormalized square QPSK
# C_qpsk_raw: Four unnormalized QPSK points, with I and Q each equal to +/-1.
C_qpsk_raw = np.array([1+1j, -1+1j, -1-1j, 1-1j])

# Unnormalized square 16-QAM with I,Q in {-3,-1,+1,+3}
# levels: Allowed raw amplitudes on either 16-QAM axis: [-3, -1, 1, 3].
levels = np.array([-3, -1, +1, +3])
# Nested comprehensions are nested loops: for each q, iterate over all i.
# levels[::-1] visits Q levels in reverse order. 1j is the imaginary unit, like
# std::complex<double>(0,1).
# Each point i + 1j*q stores I in .real and Q in .imag; there are 4*4 = 16 points.
C_qam16_raw = np.array([i + 1j*q for q in levels[::-1] for i in levels])

# Plot the raw ideal constellations
# fig: Figure containing the three panels; plotting object only.
# ax: Array of three Axes objects, indexed 0, 1, 2.
fig, ax = plt.subplots(1, 3, figsize=(12, 4))

# Plot explicit real (I) coordinates horizontally and imaginary (Q) coordinates vertically.
ax[0].scatter(C_bpsk.real, C_bpsk.imag, s=90)
ax[0].set_title("BPSK")
ax[0].set_xlim([-1.5, 1.5]); ax[0].set_ylim([-1.5, 1.5])

# Plot explicit real (I) coordinates horizontally and imaginary (Q) coordinates vertically.
ax[1].scatter(C_qpsk_raw.real, C_qpsk_raw.imag, s=90)
ax[1].set_title("Raw QPSK")
ax[1].set_xlim([-1.7, 1.7]); ax[1].set_ylim([-1.7, 1.7])

# Plot explicit real (I) coordinates horizontally and imaginary (Q) coordinates vertically.
ax[2].scatter(C_qam16_raw.real, C_qam16_raw.imag, s=70)
ax[2].set_title("Raw 16-QAM")
ax[2].set_xlim([-4, 4]); ax[2].set_ylim([-4, 4])

# Repeat the indented block for each item in this sequence or range.
# a: Current Axes object in the formatting loop; not a symbol amplitude.
for a in ax:
    a.axhline(0); a.axvline(0)
    # Keep equal physical scale on I and Q so rotations/distances are not visually distorted.
    a.set_aspect("equal", adjustable="box")
    a.set_xlabel("I"); a.set_ylabel("Q")
    a.grid(True)

# Adjust subplot spacing so labels fit; this is presentation only.
plt.tight_layout()
# Display the completed figure in Jupyter; this does not compute another transform or change
# the data.
plt.show()

## 4. Compute constellation mean and average symbol energy


---
### Instructor guide: 4. Constellation mean and symbol energy

**Purpose:** Compute the theoretical equal-probability constellation mean and average squared magnitude.

**Ask before running:** Do symmetric point sets have zero mean? Are all three raw energies already equal?

**What to expect:** Means are zero. Raw average energies are 1 for BPSK, 2 for QPSK, and 10 for 16-QAM.

**Teaching notes / model boundaries:** The uniform arithmetic mean assumes each constellation point is equally likely. For a point I+jQ, squared magnitude is I^2+Q^2. The energy is the discrete symbol energy in this model; identifying it with waveform energy assumes the appropriate unit-energy pulse convention.

#### Variable reference for this block

| Variable | Meaning / units / role in this block |
|---|---|
| `C` | Local function argument: the complete constellation array. |
| `mean_bpsk` | Average of the two raw BPSK points: 0j. |
| `mean_qpsk` | Average of the four raw QPSK points: 0j. |
| `mean_qam16` | Average of the sixteen raw 16-QAM points: 0j. |
| `energy_bpsk` | Raw equal-probability symbol energy: 1. |
| `energy_qpsk` | Raw equal-probability symbol energy: 2. |
| `energy_qam16` | Raw equal-probability symbol energy: 10. |

**Live-demo cue:** Explain the prediction and result first. Use the detailed comments below only when a code operation needs clarification.


In [ ]:
# ========================================================================================
# INSTRUCTOR GUIDE | 4. Constellation mean and symbol energy
# Purpose: Compute the theoretical equal-probability constellation mean and average squared
# magnitude.
# Original executable statements below are unchanged. Expanded variable explanations appear in
# the Markdown immediately above this cell.
# Read the calculation comments as preparation; the live demo can focus on prediction and the
# resulting plot.
# ========================================================================================

# 4. Compute constellation mean and average symbol energy

# Compute the mean of equally likely points in a supplied constellation.
# The indented body runs only when called. Arguments and assignments inside this function are
# local.
def constellation_mean(C):
    # Average the complex coordinates across all allowed points. Real and imaginary components
    # are averaged separately.
    # For the symmetric, equally likely constellations used here, both components cancel.
    return np.mean(C)

# Compute the average squared magnitude of the supplied constellation points.
# The indented body runs only when called. Arguments and assignments inside this function are
# local.
def average_symbol_energy(C):
    # abs(C) is distance from the I/Q origin, NOT just the real component. Squaring gives
    # I^2+Q^2.
    # Average those squared distances over the alphabet. This is not the same as
    # abs(mean(C))**2.
    return np.mean(np.abs(C)**2)

# mean_bpsk: Average of the two raw BPSK points: 0j.
mean_bpsk = constellation_mean(C_bpsk)
# mean_qpsk: Average of the four raw QPSK points: 0j.
mean_qpsk = constellation_mean(C_qpsk_raw)
# mean_qam16: Average of the sixteen raw 16-QAM points: 0j.
mean_qam16 = constellation_mean(C_qam16_raw)

# energy_bpsk: Raw equal-probability symbol energy: 1.
energy_bpsk = average_symbol_energy(C_bpsk)
# energy_qpsk: Raw equal-probability symbol energy: 2.
energy_qpsk = average_symbol_energy(C_qpsk_raw)
# energy_qam16: Raw equal-probability symbol energy: 10.
energy_qam16 = average_symbol_energy(C_qam16_raw)

# Print a teaching label or takeaway; no signal-processing state changes here.
print("RAW CONSTELLATION STATISTICS")
# Report calculated values. f-string precision/sign specifiers affect text only, not the
# stored numbers.
print(f"BPSK:   mean = {mean_bpsk.real:.2f}{mean_bpsk.imag:+.2f}j, Es = {energy_bpsk:.2f}")
# Report calculated values. f-string precision/sign specifiers affect text only, not the
# stored numbers.
print(f"QPSK:   mean = {mean_qpsk.real:.2f}{mean_qpsk.imag:+.2f}j, Es = {energy_qpsk:.2f}")
# Report calculated values. f-string precision/sign specifiers affect text only, not the
# stored numbers.
print(f"16-QAM: mean = {mean_qam16.real:.2f}{mean_qam16.imag:+.2f}j, Es = {energy_qam16:.2f}")

## 5. Normalize each constellation to unit average symbol energy


---
### Instructor guide: 5. Normalize to unit average energy

**Purpose:** Rescale each alphabet so comparisons do not simply reward a larger original signal energy.

**Ask before running:** Must amplitudes be divided by energy, or by its square root?

**What to expect:** Scale factors are 1, 1/sqrt(2), and 1/sqrt(10); all normalized average energies are 1.

**Teaching notes / model boundaries:** Energy scales as the square of an amplitude multiplier: E_new = alpha^2 E_old. Normalizing 16-QAM does not put every point on a unit circle; only its average squared distance is one.

#### Variable reference for this block

| Variable | Meaning / units / role in this block |
|---|---|
| `alpha_bpsk` | BPSK amplitude multiplier 1/sqrt(energy_bpsk), equal to 1. |
| `alpha_qpsk` | QPSK amplitude multiplier 1/sqrt(2). |
| `alpha_qam16` | 16-QAM amplitude multiplier 1/sqrt(10). |
| `C_bpsk_norm` | BPSK alphabet after unit-average-energy scaling. |
| `C_qpsk_norm` | QPSK alphabet after unit-average-energy scaling. |
| `C_qam16_norm` | 16-QAM alphabet after unit-average-energy scaling. |

**Live-demo cue:** Explain the prediction and result first. Use the detailed comments below only when a code operation needs clarification.


In [ ]:
# ========================================================================================
# INSTRUCTOR GUIDE | 5. Normalize to unit average energy
# Purpose: Rescale each alphabet so comparisons do not simply reward a larger original signal
# energy.
# Original executable statements below are unchanged. Expanded variable explanations appear in
# the Markdown immediately above this cell.
# Read the calculation comments as preparation; the live demo can focus on prediction and the
# resulting plot.
# ========================================================================================

# 5. Normalize each constellation to unit average symbol energy

# alpha_bpsk: BPSK amplitude multiplier 1/sqrt(energy_bpsk), equal to 1.
alpha_bpsk = 1 / np.sqrt(energy_bpsk)
# alpha_qpsk: QPSK amplitude multiplier 1/sqrt(2).
alpha_qpsk = 1 / np.sqrt(energy_qpsk)
# alpha_qam16: 16-QAM amplitude multiplier 1/sqrt(10).
alpha_qam16 = 1 / np.sqrt(energy_qam16)

# C_bpsk_norm: BPSK alphabet after unit-average-energy scaling.
C_bpsk_norm = alpha_bpsk * C_bpsk
# C_qpsk_norm: QPSK alphabet after unit-average-energy scaling.
C_qpsk_norm = alpha_qpsk * C_qpsk_raw
# C_qam16_norm: 16-QAM alphabet after unit-average-energy scaling.
C_qam16_norm = alpha_qam16 * C_qam16_raw

# Print a teaching label or takeaway; no signal-processing state changes here.
print("NORMALIZATION SCALE FACTORS")
# Report calculated values. f-string precision/sign specifiers affect text only, not the
# stored numbers.
print(f"BPSK:   alpha = {alpha_bpsk:.6f}")
# Report calculated values. f-string precision/sign specifiers affect text only, not the
# stored numbers.
print(f"QPSK:   alpha = {alpha_qpsk:.6f} = 1/sqrt(2)")
# Report calculated values. f-string precision/sign specifiers affect text only, not the
# stored numbers.
print(f"16-QAM: alpha = {alpha_qam16:.6f} = 1/sqrt(10)")

# Print a teaching label or takeaway; no signal-processing state changes here.
print("\nENERGY AFTER NORMALIZATION")
# Report calculated values. f-string precision/sign specifiers affect text only, not the
# stored numbers.
print(f"BPSK:   Es = {average_symbol_energy(C_bpsk_norm):.6f}")
# Report calculated values. f-string precision/sign specifiers affect text only, not the
# stored numbers.
print(f"QPSK:   Es = {average_symbol_energy(C_qpsk_norm):.6f}")
# Report calculated values. f-string precision/sign specifiers affect text only, not the
# stored numbers.
print(f"16-QAM: Es = {average_symbol_energy(C_qam16_norm):.6f}")

## 6. Plot raw and normalized 16-QAM


---
### Instructor guide: 6. Compare raw and normalized 16-QAM

**Purpose:** Make the normalization visible without changing which points belong to the alphabet.

**Ask before running:** Will normalization remove any points or change their relative geometry?

**What to expect:** Both panels have the same sixteen-point arrangement. Numerical axis limits show the smaller normalized coordinates.

**Teaching notes / model boundaries:** The two panels use different numerical axis limits. Similar visual size on the page does not mean the amplitudes are unchanged. Read the tick labels and the printed energy in each title.

#### Variable reference for this block

| Variable | Meaning / units / role in this block |
|---|---|
| `fig` | Figure for the raw/normalized comparison. |
| `ax` | Two Axes objects: ax[0] raw, ax[1] normalized. |
| `a` | Current plotting axis in the common formatting loop. |

**Live-demo cue:** Explain the prediction and result first. Use the detailed comments below only when a code operation needs clarification.


In [ ]:
# ========================================================================================
# INSTRUCTOR GUIDE | 6. Compare raw and normalized 16-QAM
# Purpose: Make the normalization visible without changing which points belong to the
# alphabet.
# Original executable statements below are unchanged. Expanded variable explanations appear in
# the Markdown immediately above this cell.
# Read the calculation comments as preparation; the live demo can focus on prediction and the
# resulting plot.
# ========================================================================================

# 6. Plot raw and normalized 16-QAM

# fig: Figure for the raw/normalized comparison.
# ax: Two Axes objects: ax[0] raw, ax[1] normalized.
fig, ax = plt.subplots(1, 2, figsize=(10, 4.8))

# Plot explicit real (I) coordinates horizontally and imaginary (Q) coordinates vertically.
ax[0].scatter(C_qam16_raw.real, C_qam16_raw.imag, s=70)
ax[0].set_title(f"Raw 16-QAM: $E_s$ = {energy_qam16:.1f}")
ax[0].set_xlim([-4, 4]); ax[0].set_ylim([-4, 4])

# Plot explicit real (I) coordinates horizontally and imaginary (Q) coordinates vertically.
ax[1].scatter(C_qam16_norm.real, C_qam16_norm.imag, s=70)
ax[1].set_title(f"Normalized 16-QAM: $E_s$ = {average_symbol_energy(C_qam16_norm):.1f}")
ax[1].set_xlim([-1.2, 1.2]); ax[1].set_ylim([-1.2, 1.2])

# Repeat the indented block for each item in this sequence or range.
# a: Current plotting axis in the common formatting loop.
for a in ax:
    a.axhline(0); a.axvline(0)
    # Keep equal physical scale on I and Q so rotations/distances are not visually distorted.
    a.set_aspect("equal", adjustable="box")
    a.set_xlabel("I"); a.set_ylabel("Q")
    a.grid(True)

# Adjust subplot spacing so labels fit; this is presentation only.
plt.tight_layout()
# Display the completed figure in Jupyter; this does not compute another transform or change
# the data.
plt.show()

## 7. Verify mean and energy using finite random symbol records


---
### Instructor guide: 7. Finite random symbol records

**Purpose:** Select random alphabet entries and compare finite-record statistics with the full-alphabet results.

**Ask before running:** Will 5000 randomly chosen symbols have exactly zero mean? Will both energy estimates be exactly one?

**What to expect:** The means are close to zero. QPSK energy is exactly one in this model because every QPSK point has unit energy; finite 16-QAM energy fluctuates near one.

**Teaching notes / model boundaries:** An index array chooses entries, not frequencies or samples per symbol. A finite realization need not use each point equally often. The constellation energy is a model average; the record energy is a sample average.

#### Variable reference for this block

| Variable | Meaning / units / role in this block |
|---|---|
| `N_symbols` | Number of randomly selected symbols: 5000. |
| `qpsk_index` | Random integer alphabet indices 0 through 3, one per QPSK symbol. |
| `qam16_index` | Random integer alphabet indices 0 through 15, one per 16-QAM symbol. |
| `qpsk_record` | Time-ordered sequence of chosen normalized QPSK symbols. |
| `qam16_record` | Time-ordered sequence of chosen normalized 16-QAM symbols. |

**Live-demo cue:** Explain the prediction and result first. Use the detailed comments below only when a code operation needs clarification.


In [ ]:
# ========================================================================================
# INSTRUCTOR GUIDE | 7. Finite random symbol records
# Purpose: Select random alphabet entries and compare finite-record statistics with the
# full-alphabet results.
# Original executable statements below are unchanged. Expanded variable explanations appear in
# the Markdown immediately above this cell.
# Read the calculation comments as preparation; the live demo can focus on prediction and the
# resulting plot.
# ========================================================================================

# 7. Verify mean and energy using finite random symbol records
# A finite sequence estimates the theoretical constellation statistics.

# N_symbols: Number of randomly selected symbols: 5000.
N_symbols = 5000

# qpsk_index: Random integer alphabet indices 0 through 3, one per QPSK symbol.
# integers uses an exclusive upper bound; size determines the number/shape of draws.
qpsk_index = rng.integers(0, len(C_qpsk_norm), size=N_symbols)
# qam16_index: Random integer alphabet indices 0 through 15, one per 16-QAM symbol.
# integers uses an exclusive upper bound; size determines the number/shape of draws.
qam16_index = rng.integers(0, len(C_qam16_norm), size=N_symbols)

# Integer-array indexing gathers a constellation point for every random index.
# C++ mental model: qpsk_record[n] = C_qpsk_norm[qpsk_index[n]].
qpsk_record = C_qpsk_norm[qpsk_index]
# Apply the same indexed lookup to the sixteen-point alphabet.
# Outer 16-QAM points have more energy, so the finite-record energy depends slightly on which
# points were drawn.
qam16_record = C_qam16_norm[qam16_index]

# Print a teaching label or takeaway; no signal-processing state changes here.
print("FINITE RANDOM SYMBOL RECORDS")
# Report calculated values. f-string precision/sign specifiers affect text only, not the
# stored numbers.
print(f"QPSK record mean:     {np.mean(qpsk_record).real:.4f}{np.mean(qpsk_record).imag:+.4f}j")
# Report calculated values. f-string precision/sign specifiers affect text only, not the
# stored numbers.
print(f"QPSK record energy:   {np.mean(np.abs(qpsk_record)**2):.4f}")
# Report calculated values. f-string precision/sign specifiers affect text only, not the
# stored numbers.
print(f"16-QAM record mean:   {np.mean(qam16_record).real:.4f}{np.mean(qam16_record).imag:+.4f}j")
# Report calculated values. f-string precision/sign specifiers affect text only, not the
# stored numbers.
print(f"16-QAM record energy: {np.mean(np.abs(qam16_record)**2):.4f}")

## 8. Add AWGN and display measured constellation clouds


---
### Instructor guide: 8. Add complex AWGN

**Purpose:** Add complex noise to each normalized symbol record and inspect received clouds at the same target signal-to-noise ratio.

**Ask before running:** At equal average energy, which alphabet has more tightly spaced decision candidates?

**What to expect:** QPSK and 16-QAM each form clouds around their ideal points. The sixteen-point alphabet is more crowded. This cell does not make decisions or measure BER.

**Teaching notes / model boundaries:** The noise variance is a target ensemble value, not an exact finite-realization guarantee. This SNR is per symbol/sample in the discrete model, not Eb/N0. The first 1500 points are plotted but all 5000 are used for the printed noise-power checks.

#### Variable reference for this block

| Variable | Meaning / units / role in this block |
|---|---|
| `SNR_dB` | Target signal-power/noise-power ratio in dB: 12. |
| `SNR_linear` | Linear ratio 10**(SNR_dB/10), about 15.85. |
| `qpsk_signal_power` | Measured mean squared magnitude of the QPSK record, equal to 1 here. |
| `qam16_signal_power` | Measured mean squared magnitude of the finite 16-QAM record, near 1. |
| `qpsk_noise_power` | Target total complex noise power for QPSK: signal power / SNR_linear. |
| `qam16_noise_power` | Target total complex noise power for 16-QAM. |
| `qpsk_noise` | Complex Gaussian noise array; I and Q each get half the target variance. |
| `qam16_noise` | Separate complex Gaussian noise array for the 16-QAM example. |
| `qpsk_received` | Original QPSK symbols plus their noise. |
| `qam16_received` | Original 16-QAM symbols plus their noise. |
| `fig` | Figure holding two constellation panels. |
| `ax` | Panel objects, not signal arrays. |
| `a` | Current panel in the formatting loop. |

**Live-demo cue:** Explain the prediction and result first. Use the detailed comments below only when a code operation needs clarification.


In [ ]:
# ========================================================================================
# INSTRUCTOR GUIDE | 8. Add complex AWGN
# Purpose: Add complex noise to each normalized symbol record and inspect received clouds at
# the same target signal-to-noise ratio.
# Original executable statements below are unchanged. Expanded variable explanations appear in
# the Markdown immediately above this cell.
# Read the calculation comments as preparation; the live demo can focus on prediction and the
# resulting plot.
# ========================================================================================

# 8. Add AWGN and display measured constellation clouds
# Both constellations have unit average symbol energy before noise is added.
# Therefore, using the same SNR gives a fair equal-energy visual comparison.

# SNR_dB: Target signal-power/noise-power ratio in dB: 12.
SNR_dB = 12
# SNR_linear: Linear ratio 10**(SNR_dB/10), about 15.85.
# A power ratio in dB converts to linear with 10**(dB/10).
SNR_linear = 10**(SNR_dB/10)

# qpsk_signal_power: Measured mean squared magnitude of the QPSK record, equal to 1 here.
# With no axis argument, mean averages all entries in the supplied array, returning a scalar.
qpsk_signal_power = np.mean(np.abs(qpsk_record)**2)
# qam16_signal_power: Measured mean squared magnitude of the finite 16-QAM record, near 1.
# With no axis argument, mean averages all entries in the supplied array, returning a scalar.
qam16_signal_power = np.mean(np.abs(qam16_record)**2)

# qpsk_noise_power: Target total complex noise power for QPSK: signal power / SNR_linear.
qpsk_noise_power = qpsk_signal_power / SNR_linear
# qam16_noise_power: Target total complex noise power for 16-QAM.
qam16_noise_power = qam16_signal_power / SNR_linear

# Generate independent real Gaussian draws for I and Q. Each raw part has variance one.
# Multiplying by sqrt(Pnoise/2) makes their variances Pnoise/2, so E[abs(w)**2] = Pnoise.
# .shape requests the same array dimensions as the signal; it does not copy the signal values.
qpsk_noise = np.sqrt(qpsk_noise_power/2) * (
    rng.standard_normal(qpsk_record.shape) + 1j*rng.standard_normal(qpsk_record.shape)
)
# Use fresh random I/Q noise samples with the 16-QAM target power.
# Scaling comes from the measured clean-record energy, so both examples use the same target
# SNR despite finite-record energy differences.
qam16_noise = np.sqrt(qam16_noise_power/2) * (
    rng.standard_normal(qam16_record.shape) + 1j*rng.standard_normal(qam16_record.shape)
)

# qpsk_received: Original QPSK symbols plus their noise.
qpsk_received = qpsk_record + qpsk_noise
# qam16_received: Original 16-QAM symbols plus their noise.
qam16_received = qam16_record + qam16_noise

# fig: Figure holding two constellation panels.
# ax: Panel objects, not signal arrays.
fig, ax = plt.subplots(1, 2, figsize=(11, 4.8))

# Plot explicit real (I) coordinates horizontally and imaginary (Q) coordinates vertically.
ax[0].scatter(qpsk_received[:1500].real, qpsk_received[:1500].imag, s=8, alpha=0.75)
# Plot explicit real (I) coordinates horizontally and imaginary (Q) coordinates vertically.
ax[0].scatter(C_qpsk_norm.real, C_qpsk_norm.imag, s=100, marker='x', linewidths=2)
ax[0].set_title(f"QPSK at {SNR_dB:g} dB SNR")
ax[0].set_xlim([-1.7, 1.7]); ax[0].set_ylim([-1.7, 1.7])

# Plot explicit real (I) coordinates horizontally and imaginary (Q) coordinates vertically.
ax[1].scatter(qam16_received[:1500].real, qam16_received[:1500].imag, s=8, alpha=0.75)
# Plot explicit real (I) coordinates horizontally and imaginary (Q) coordinates vertically.
ax[1].scatter(C_qam16_norm.real, C_qam16_norm.imag, s=100, marker='x', linewidths=2)
ax[1].set_title(f"16-QAM at {SNR_dB:g} dB SNR")
ax[1].set_xlim([-1.7, 1.7]); ax[1].set_ylim([-1.7, 1.7])

# Repeat the indented block for each item in this sequence or range.
# a: Current panel in the formatting loop.
for a in ax:
    a.axhline(0); a.axvline(0)
    # Keep equal physical scale on I and Q so rotations/distances are not visually distorted.
    a.set_aspect("equal", adjustable="box")
    a.set_xlabel("I"); a.set_ylabel("Q")
    a.grid(True)

# Adjust subplot spacing so labels fit; this is presentation only.
plt.tight_layout()
# Display the completed figure in Jupyter; this does not compute another transform or change
# the data.
plt.show()

# Print a teaching label or takeaway; no signal-processing state changes here.
print("NOISY CONSTELLATION EXAMPLE")
# Report calculated values. f-string precision/sign specifiers affect text only, not the
# stored numbers.
print(f"SNR used for both normalized constellations: {SNR_dB:.1f} dB")
# Report calculated values. f-string precision/sign specifiers affect text only, not the
# stored numbers.
print(f"Measured QPSK noise power:   {np.mean(np.abs(qpsk_noise)**2):.4f}")
# Report calculated values. f-string precision/sign specifiers affect text only, not the
# stored numbers.
print(f"Measured 16-QAM noise power: {np.mean(np.abs(qam16_noise)**2):.4f}")

## 9. Demonstration summary


---
### Instructor guide: 9. Recap

**Purpose:** Summarize the transition from bits to normalized symbols to noisy observations.

**Ask before running:** Can students distinguish the alphabet, a symbol record, and a received cloud?

**What to expect:** This cell only prints the original takeaways; no new signals are created.

**Teaching notes / model boundaries:** The zero-mean and energy statements refer to the equally likely alphabets in this notebook. Week 4 introduces decisions; this notebook does not implement a demodulator.


**Live-demo cue:** Explain the prediction and result first. Use the detailed comments below only when a code operation needs clarification.


In [ ]:
# ========================================================================================
# INSTRUCTOR GUIDE | 9. Recap
# Purpose: Summarize the transition from bits to normalized symbols to noisy observations.
# Original executable statements below are unchanged. Expanded variable explanations appear in
# the Markdown immediately above this cell.
# Read the calculation comments as preparation; the live demo can focus on prediction and the
# resulting plot.
# ========================================================================================

# 9. Demonstration summary

# Print a teaching label or takeaway; no signal-processing state changes here.
print("WEEK 3 SUMMARY")
# Print a teaching label or takeaway; no signal-processing state changes here.
print("1. M = 2^b and b = log2(M).")
# Print a teaching label or takeaway; no signal-processing state changes here.
print("2. Bit groups map to symbols from a finite constellation.")
# Print a teaching label or takeaway; no signal-processing state changes here.
print("3. Complex symbols have I and Q coordinates.")
# Print a teaching label or takeaway; no signal-processing state changes here.
print("4. Symmetric BPSK, QPSK, and 16-QAM have zero mean.")
# Print a teaching label or takeaway; no signal-processing state changes here.
print("5. Raw BPSK, QPSK, and 16-QAM energies here are 1, 2, and 10.")
# Print a teaching label or takeaway; no signal-processing state changes here.
print("6. Unit-energy normalization uses alpha = 1/sqrt(Es).")
# Print a teaching label or takeaway; no signal-processing state changes here.
print("7. Noise turns ideal constellation points into received clouds.")
# Print a teaching label or takeaway; no signal-processing state changes here.
print("8. Week 4 asks how the receiver decides which ideal point was sent.")



---
### Optional syntax references

The signal-processing examples and numeric settings above come from the supplied notebook. The added commentary is an instructor explanation of those operations, not an independent replacement for the course materials. Official API references for unfamiliar syntax:

- [NumPy indexing and slices](https://numpy.org/doc/stable/user/basics.indexing.html)
- [NumPy broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [NumPy FFT conventions and normalization](https://numpy.org/doc/stable/reference/routines.fft.html)
- [NumPy pseudoinverse](https://numpy.org/doc/stable/reference/generated/numpy.linalg.pinv.html)
- [NumPy linear interpolation](https://numpy.org/doc/stable/reference/generated/numpy.interp.html)
- [Matplotlib plot argument conventions](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html)

The comments were added without changing executable tokens or abstract syntax trees. Numerical and plot-data equivalence checks are documented in the accompanying validation report.
